In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch
import cv2
import json
import matplotlib.pyplot as plt
from pathlib import Path
import sys
from typing import Any

from project_root import PROJECT_ROOT,DATASETS_ROOT

import fiftyone as fo
import fiftyone.utils.torch as fout


import torchvision as tv

no_grad_guard = torch.no_grad()

In [ ]:
# dataset_name =         "zoo-elephants-identity-tracks"
# ds =fo.load_dataset(dataset_name)


def make_ds():
    return fo.Dataset.from_dir(
        dataset_type=fo.types.ImageDirectory,
        dataset_dir="/home/dherrera/data/elephants/certainty/v4",
        persistent=False,
    )
    # return fo.Dataset.from_dir(
    #     dataset_type=fo.types.ImageClassificationDirectoryTree,
    #     dataset_dir="/home/dherrera/data/elephants/behaviour/sleep_v2",
    #     persistent=False,
    # )


ds = make_ds()
# classes = sorted(ds.classes["ground_truth"])
# print(classes)
print(ds.stats(include_media=True))

 100% |███████████████| 8017/8017 [530.9ms elapsed, 0s remaining, 15.1K samples/s]      
Computing metadata...
 100% |███████████████| 8017/8017 [3.6s elapsed, 0s remaining, 2.1K samples/s]      
{'samples_count': 8017, 'samples_bytes': 2044917, 'samples_size': '2.0MB', 'media_bytes': 51356006, 'media_size': '49.0MB', 'total_bytes': 53400923, 'total_size': '50.9MB'}


In [5]:
# Compute uniqueness
import fiftyone.brain as fob

session = fo.launch_app(ds, auto=False)
session.open_tab()

Session launched. Run `session.show()` to open the App in a cell output.


<IPython.core.display.Javascript object>

In [ ]:
from tqdm.auto import tqdm

dup_result = fob.compute_near_duplicates(ds)
print(len(dup_result.duplicate_ids))
session.view = dup_result.duplicates_view()

DELETE_DUPLICATES = False
if DELETE_DUPLICATES:
    while len(dup_result.duplicate_ids) > 0:
        print(f"Deleting {len(dup_result.duplicate_ids)} duplicate files")
        for id in tqdm(dup_result.duplicate_ids):
            filepath = ds[id]["filepath"]
            Path.unlink(filepath)
        ds = make_ds()
        dup_result = fob.compute_near_duplicates(ds)
        print(len(dup_result.duplicate_ids))

Computing embeddings...
 100% |███████████████| 8017/8017 [21.7s elapsed, 0s remaining, 317.8 samples/s]      
Computing duplicate samples...
Generating index for 8017 embeddings...
Index complete
Generating neighbors graph for 2641 embeddings...
Index complete
Duplicates computation complete
5376
Deleting 5376 duplicate files


  0%|          | 0/5376 [00:00<?, ?it/s]

 100% |███████████████| 2641/2641 [169.6ms elapsed, 0s remaining, 15.7K samples/s]  
Computing embeddings...
 100% |███████████████| 2641/2641 [6.6s elapsed, 0s remaining, 314.8 samples/s]       
Computing duplicate samples...
Generating index for 2641 embeddings...
Index complete
Generating neighbors graph for 2466 embeddings...
Index complete
Duplicates computation complete
175
Deleting 175 duplicate files


  0%|          | 0/175 [00:00<?, ?it/s]

 100% |███████████████| 2466/2466 [256.9ms elapsed, 0s remaining, 9.6K samples/s]   
Computing embeddings...
 100% |███████████████| 2466/2466 [6.4s elapsed, 0s remaining, 329.1 samples/s]       
Computing duplicate samples...
Generating index for 2466 embeddings...
Index complete
Generating neighbors graph for 2431 embeddings...
Index complete
Duplicates computation complete
35
Deleting 35 duplicate files


...

 100% |███████████████| 2424/2424 [183.5ms elapsed, 0s remaining, 13.3K samples/s]  
Computing embeddings...
 100% |███████████████| 2424/2424 [7.3s elapsed, 0s remaining, 314.8 samples/s]       
Computing duplicate samples...
Generating index for 2424 embeddings...
Index complete
Generating neighbors graph for 2423 embeddings...
Index complete
Duplicates computation complete
1
Deleting 1 duplicate files


  0%|          | 0/1 [00:00<?, ?it/s]

 100% |███████████████| 2423/2423 [259.9ms elapsed, 0s remaining, 9.3K samples/s]      
Computing embeddings...
 100% |███████████████| 2423/2423 [7.5s elapsed, 0s remaining, 316.4 samples/s]       
Computing duplicate samples...
Generating index for 2423 embeddings...
Index complete
Generating neighbors graph for 2422 embeddings...
Index complete
Duplicates computation complete
1
Deleting 1 duplicate files


  0%|          | 0/1 [00:00<?, ?it/s]

 100% |███████████████| 2422/2422 [201.9ms elapsed, 0s remaining, 12.0K samples/s]    
Computing embeddings...
 100% |███████████████| 2422/2422 [7.3s elapsed, 0s remaining, 338.3 samples/s]       
Computing duplicate samples...
Generating index for 2422 embeddings...
Index complete
Duplicates computation complete
0


In [8]:
from fiftyone import ViewField as F

THRESHOLD = 0.01
fob.compute_uniqueness(ds)
# session.view = ds.sort_by("uniqueness", reverse=True)

ds_bad = ds.match(F("uniqueness") < THRESHOLD)
print(f"Samples to drop: {len(ds_bad)}")
session.view = ds_bad

from tqdm.auto import tqdm

DELETE_BAD = False
if DELETE_BAD:
    while len(ds_bad) > 0:
        print(f"Deleting {len(ds_bad)} low-uniqueness files")
        for sample in tqdm(ds_bad):
            filepath = sample["filepath"]
            Path.unlink(filepath)
        ds = make_ds()
        fob.compute_uniqueness(ds)
        ds_bad = ds.match(F("uniqueness") < THRESHOLD)

session.view = ds_bad

Computing embeddings...
 100% |███████████████| 1892/1892 [553.4ms elapsed, 0s remaining, 3.4K samples/s]      
Computing uniqueness...
Generating index for 1892 embeddings...
Index complete
Uniqueness computation complete
Samples to drop: 0


In [13]:
session.view = ds.match(F("uniqueness") >= 0.01)